# __PR 14_1__

__Importamos las librerias__

In [1]:
import requests
import json

from pymongo import MongoClient

__Definimos la url de la API__

In [ ]:
url_busqueda = "https://www.thecocktaildb.com/api/json/v1/1/search.php"
url_filtro = "https://www.thecocktaildb.com/api/json/v1/1/filter.php"

url_bbdd = "uri"

# 1. Apartado A

_Crea una base de datos y una colección en tu clúster de Atlas MongoDB para almacenar los datos._

* Nos conectamos a al cliente

In [3]:
cliente = MongoClient(url_bbdd)
print(cliente.state)


Database(MongoClient(host=['ac-oouwvxl-shard-00-02.0251v4q.mongodb.net:27017', 'ac-oouwvxl-shard-00-01.0251v4q.mongodb.net:27017', 'ac-oouwvxl-shard-00-00.0251v4q.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, authsource='admin', replicaset='atlas-hymc7m-shard-0', tls=True), 'state')


* Eliminamos la base de datos, si existe

In [4]:
if "cocktail" in cliente.list_database_names():
    cliente.drop_database("cocktail")

* Accedemos a la base de datos y a la colección

In [5]:
db = cliente["cocktail"]

col = db["bebidas"]

* Llamamos a la api

In [ ]:
respuesta = requests.get(url_busqueda, params={"s": "margarita"})

* Extraemos la consulta

In [7]:
consulta = json.loads(respuesta.content)["drinks"]

* Definimos los campos a mantener

In [8]:
campos = [
    "idDrink",
    "strDrink",
    "strDrinkAlternate",
    "strTags",
    "strVideo",
    "strCategory",
    "strIBA",
    "strAlcoholic",
    "strGlass",
    "strInstructions",
    "strInstructionsES",
    "strInstructionsDE",
    "strInstructionsFR",
    "strInstructionsIT",
    "strInstructionsZH-HANS",
    "strInstructionsZH-HANT",
    "strDrinkThumb",
    "strImageSource",
    "strImageAttribution",
    "strCreativeCommonsConfirmed",
    "dateModified"
]

* Generamos el resultado esperado

In [9]:
cocktailsGestionados = []
for cocktail in consulta:
    ingredientes = []

    for i in range(1, 16):
        ingrediente = cocktail[f"strIngredient{i}"]
        cantidad = cocktail[f"strMeasure{i}"]

        if ingrediente is not None:
            ingredientes.append({"name": ingrediente, "measure": cantidad})

    gestionado = {}
    for i in campos:
        gestionado[i] = cocktail[i]
    
    gestionado["ingredients"] = ingredientes

    cocktailsGestionados.append(gestionado)

* Introducimos los datos

In [10]:
insercion = col.insert_many(cocktailsGestionados)

* Comprobamos la introducción

In [11]:
for i in col.find().limit(10):
    print(i)

{'_id': ObjectId('69d75bb42628697bbab6cd54'), 'idDrink': '11728', 'strDrink': 'Martini', 'strDrinkAlternate': None, 'strTags': None, 'strVideo': 'https://www.youtube.com/watch?v=ApMR3IWYZHI', 'strCategory': 'Cocktail', 'strIBA': 'Unforgettables', 'strAlcoholic': 'Alcoholic', 'strGlass': 'Cocktail glass', 'strInstructions': 'Straight: Pour all ingredients into mixing glass with ice cubes. Stir well. Strain in chilled martini cocktail glass. Squeeze oil from lemon peel onto the drink, or garnish with olive.', 'strInstructionsES': 'Derecho: Vierta todos los ingredientes en un vaso mezclador con cubitos de hielo. Remover bien. Colar en una copa de martini fría. Exprima el aceite de la cáscara de limón sobre la bebida, o decore con aceitunas.', 'strInstructionsDE': 'Direkt: Alle Zutaten in ein Mischglas mit Eiswürfeln geben. Gut umrühren. In einem gekühlten Martini-Cocktailglas abseihen. Den Saft aus der Zitronenschale auf das Getränk drücken oder mit Olive garnieren.', 'strInstructionsFR':

# 2. Apartado B

_Una vez almacenados los cócteles en la colección de Atlas MongoDB realiza las siguientes consultas_

1. Lista todos los cócteles que contienen alcohol

In [12]:
query = col.find({"strAlcoholic": "Alcoholic"})

for i in query:
    print(i)

{'_id': ObjectId('69d75bb42628697bbab6cd54'), 'idDrink': '11728', 'strDrink': 'Martini', 'strDrinkAlternate': None, 'strTags': None, 'strVideo': 'https://www.youtube.com/watch?v=ApMR3IWYZHI', 'strCategory': 'Cocktail', 'strIBA': 'Unforgettables', 'strAlcoholic': 'Alcoholic', 'strGlass': 'Cocktail glass', 'strInstructions': 'Straight: Pour all ingredients into mixing glass with ice cubes. Stir well. Strain in chilled martini cocktail glass. Squeeze oil from lemon peel onto the drink, or garnish with olive.', 'strInstructionsES': 'Derecho: Vierta todos los ingredientes en un vaso mezclador con cubitos de hielo. Remover bien. Colar en una copa de martini fría. Exprima el aceite de la cáscara de limón sobre la bebida, o decore con aceitunas.', 'strInstructionsDE': 'Direkt: Alle Zutaten in ein Mischglas mit Eiswürfeln geben. Gut umrühren. In einem gekühlten Martini-Cocktailglas abseihen. Den Saft aus der Zitronenschale auf das Getränk drücken oder mit Olive garnieren.', 'strInstructionsFR':

2. Encuentra todos los cócteles que contienen "Tequila"

In [13]:
query = col.find({'ingredients.name':'Tequila'})

for i in query:
    print(i)

3. Devuelve solo el nombre y la categoría de los cócteles

In [14]:
query = col.find({},{"_id": 0, "strDrink": 1, "strCategory": 1})

for i in query:
    print(i)

{'strDrink': 'Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Dry Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Kiwi Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Vodka Martini', 'strCategory': 'Ordinary Drink'}
{'strDrink': 'Dirty Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Abbey Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'French Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Bellini Martini', 'strCategory': 'Ordinary Drink'}
{'strDrink': 'Espresso Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Pornstar Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Cosmopolitan Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Salted Toffee Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Passion Fruit Martini', 'strCategory': 'Cocktail'}
{'strDrink': 'Ziemes Martini Apfelsaft', 'strCategory': 'Ordinary Drink'}


4. Cócteles alcohólicos que se sirven en "Cocktail glass"

In [15]:
query = col.find({"strGlass": "Cocktail glass"})

for i in query:
    print(i)

{'_id': ObjectId('69d75bb42628697bbab6cd54'), 'idDrink': '11728', 'strDrink': 'Martini', 'strDrinkAlternate': None, 'strTags': None, 'strVideo': 'https://www.youtube.com/watch?v=ApMR3IWYZHI', 'strCategory': 'Cocktail', 'strIBA': 'Unforgettables', 'strAlcoholic': 'Alcoholic', 'strGlass': 'Cocktail glass', 'strInstructions': 'Straight: Pour all ingredients into mixing glass with ice cubes. Stir well. Strain in chilled martini cocktail glass. Squeeze oil from lemon peel onto the drink, or garnish with olive.', 'strInstructionsES': 'Derecho: Vierta todos los ingredientes en un vaso mezclador con cubitos de hielo. Remover bien. Colar en una copa de martini fría. Exprima el aceite de la cáscara de limón sobre la bebida, o decore con aceitunas.', 'strInstructionsDE': 'Direkt: Alle Zutaten in ein Mischglas mit Eiswürfeln geben. Gut umrühren. In einem gekühlten Martini-Cocktailglas abseihen. Den Saft aus der Zitronenschale auf das Getränk drücken oder mit Olive garnieren.', 'strInstructionsFR':

5. ¿Cuántos cócteles hay con "Vodka"?

In [17]:
query = col.find({'ingredients.name':'Vodka'})

print(len(list(query)))

9


6. Lista los cócteles ordenados alfabéticamente por nombre

In [18]:
query = col.find().sort("strDrink")

for i in query:
    print(i)

{'_id': ObjectId('69d75bb42628697bbab6cd59'), 'idDrink': '17223', 'strDrink': 'Abbey Martini', 'strDrinkAlternate': None, 'strTags': None, 'strVideo': None, 'strCategory': 'Cocktail', 'strIBA': None, 'strAlcoholic': 'Alcoholic', 'strGlass': 'Cocktail glass', 'strInstructions': 'Put all ingredients into a shaker and mix, then strain contents into a chilled cocktail glass.', 'strInstructionsES': 'Ponga todos los ingredientes en una coctelera, mézclelos y cuele el contenido en una copa de cóctel fría.', 'strInstructionsDE': 'Alle Zutaten in einen Shaker geben und mischen, dann den Inhalt in ein gekühltes Cocktailglas abseihen.', 'strInstructionsFR': 'Mettre tous les ingrédients dans un shaker et mélanger, puis filtrer le contenu dans un verre à cocktail réfrigéré.', 'strInstructionsIT': 'Mettere tutti gli ingredienti in uno shaker e mescolare, quindi filtrare il contenuto in una coppetta da cocktail fredda.', 'strInstructionsZH-HANS': None, 'strInstructionsZH-HANT': None, 'strDrinkThumb':

7. Obtén los 5 primeros cócteles

In [19]:
query = col.find().limit(5)

for i in query:
    print(i)

{'_id': ObjectId('69d75bb42628697bbab6cd54'), 'idDrink': '11728', 'strDrink': 'Martini', 'strDrinkAlternate': None, 'strTags': None, 'strVideo': 'https://www.youtube.com/watch?v=ApMR3IWYZHI', 'strCategory': 'Cocktail', 'strIBA': 'Unforgettables', 'strAlcoholic': 'Alcoholic', 'strGlass': 'Cocktail glass', 'strInstructions': 'Straight: Pour all ingredients into mixing glass with ice cubes. Stir well. Strain in chilled martini cocktail glass. Squeeze oil from lemon peel onto the drink, or garnish with olive.', 'strInstructionsES': 'Derecho: Vierta todos los ingredientes en un vaso mezclador con cubitos de hielo. Remover bien. Colar en una copa de martini fría. Exprima el aceite de la cáscara de limón sobre la bebida, o decore con aceitunas.', 'strInstructionsDE': 'Direkt: Alle Zutaten in ein Mischglas mit Eiswürfeln geben. Gut umrühren. In einem gekühlten Martini-Cocktailglas abseihen. Den Saft aus der Zitronenschale auf das Getränk drücken oder mit Olive garnieren.', 'strInstructionsFR':

8. Cócteles que contienen "Tequila" y "Lime juice"

In [20]:
query = col.find({'ingredients.name':{"$all":['Tequila','Lime juice']}})

for i in query:
    print(i)

9. Número de cócteles por categoría

In [21]:
query = col.aggregate([{"$group": {"_id": "$strCategory", "count": {"$sum": 1}}}])

for i in query:
    print(i)

{'_id': 'Cocktail', 'count': 11}
{'_id': 'Ordinary Drink', 'count': 3}


10. Obtener los 5 ingredientes más frecuentes

In [22]:
query = col.aggregate([{"$group":{"_id":'$ingredients.name',"count":{"$sum":1}}},{"$sort":{"count":-1}},{"$limit":5}])

for i in query:
    print(i)

{'_id': ['Gin', 'Dry Vermouth', 'Olive'], 'count': 2}
{'_id': ['Gin', 'Chocolate liqueur', 'Amaretto', 'Chocolate Sauce', 'Salted Chocolate'], 'count': 1}
{'_id': ['Vodka', 'Sugar Syrup', 'Passion fruit juice'], 'count': 1}
{'_id': ['Vodka', 'Dry Vermouth', 'Olive'], 'count': 1}
{'_id': ['Gin', 'Sweet Vermouth', 'Orange Juice', 'Angostura Bitters'], 'count': 1}


11. Cócteles sin alcohol que tengan más de 3 ingredientes

In [24]:
query = col.find({"strAlcoholic":'Non alcoholic','ingredients.name.3':{"$exists":True}})

for i in query:
    print(i)